In [ ]:
from resources.cut_off_black_segments import cut_off_black_segments
import nibabel as nib
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from resources.smooth_1d import smooth_1d
from resources.find_x import calculate_x_profile, find_x_area_center_split, find_x_area_center

In [ ]:
PROJECT_DIR = Path(r"C:\Users\abram\PycharmProjects\DrivenData_DAT_Parkinsons")
NIFTI_DIR = PROJECT_DIR / "data" / "niftis"
FILE_NAME = "08liuqdn.nii.gz"
FILE_PATH = NIFTI_DIR / FILE_NAME
LABELS_PATH = NIFTI_DIR / "train_labels_JNDlMjr.csv"

labels = pd.read_csv(LABELS_PATH)
files = sorted(NIFTI_DIR.glob("*.nii.gz"))

In [ ]:
target_map = labels.set_index("uid")["is_pathologic"].to_dict()
filter_width = 3
smooth_window = 5
n_preview = 100

In [ ]:
for i, file_path in enumerate(files[:n_preview]):
    uid = file_path.name.replace(".nii.gz", "")
    target = target_map.get(uid)
    diagnosis = "Pathological" if target == 1 else "Not pathological"

    cropped, _ = cut_off_black_segments(
        path=file_path,
        threshold=0,
        margin=1,
    )

    x_profile = calculate_x_profile(
        cropped,
        filter_width=filter_width
    )

    x_area_center = find_x_area_center(
        x_profile,
        filter_width=filter_width,
        smooth_window=smooth_window,
        edge_margin_ratio=0.10
    )

    x_center = find_x_area_center_split(
        x_profile,
        filter_width=filter_width,
        smooth_window=smooth_window,
        edge_margin_ratio=0.10,
        local_search_ratio=0.20
    )

    smoothed = smooth_1d(
        x_profile,
        window_size=smooth_window
    )

    y_mid = cropped.shape[1] // 2
    z_mid = cropped.shape[2] // 2

    fig, axes = plt.subplots(1, 3, figsize=(21, 6))

    axes[0].imshow(
        cropped[:, :, z_mid].T,
        cmap="gray",
        origin="lower"
    )
    axes[0].axvline(
    x_area_center,
    color="white",
    linestyle=":",
    linewidth=2,
    label=f"x_area={x_area_center}"
    )

    axes[0].axvline(
        x_center,
        color="lime",
        linewidth=2,
        label=f"x_split={x_center}"
    )
    axes[0].set_title(
        f"{i + 1}/{n_preview}: {file_path.name}\n"
        f"{diagnosis} | label={target}\n"
        f"Widok z góry XY dla z={z_mid}"
    )
    axes[0].set_xlabel("X")
    axes[0].set_ylabel("Y")
    axes[0].legend()

    axes[1].imshow(
        cropped[:, y_mid, :].T,
        cmap="gray",
        origin="lower"
    )
    axes[1].axvline(
    x_area_center,
    color="white",
    linestyle=":",
    linewidth=2,
    label=f"x_area={x_area_center}"
    )

    axes[1].axvline(
        x_center,
        color="lime",
        linewidth=2,
        label=f"x_split={x_center}"
    )
    axes[1].set_title(
        f"Widok XZ dla y={y_mid}"
    )
    axes[1].set_xlabel("X")
    axes[1].set_ylabel("Z")
    axes[1].legend()
    x_positions = np.arange(len(x_profile)) + filter_width // 2
    axes[2].plot(
        x_positions,
        x_profile,
        alpha=0.35,
        label="X profil oryginalny"
    )

    axes[2].plot(
        x_positions,
        smoothed,
        linewidth=2,
        label="X profil wygładzony"
    )

    axes[2].axvline(
    x_area_center,
    color="white",
    linestyle=":",
    linewidth=2,
    label=f"x_area={x_area_center}"
    )

    axes[2].axvline(
        x_center,
        color="lime",
        linewidth=2,
        label=f"x_split={x_center}"
    )

    axes[2].scatter(
        [x_center],
        [smoothed[x_center]],
        color="lime",
        zorder=5
    )

    axes[2].set_title("Profil po osi X — minimum między maksimami")
    axes[2].set_xlabel("Pozycja początku filtra X")
    axes[2].set_ylabel("Suma intensywności")
    axes[2].grid(True)
    axes[2].legend()

    plt.tight_layout()
    plt.show()